In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
#spark = SparkSession.builder.getOrCreate()
spark = SparkSession.builder.master("local[10]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DETECTOR_NAMES=["dc31","dc32"]


columns=["name","cell_size","nwires","valid_time","charge_range","time_gate","calib_runname","planes",]
detprm=[
    ("dc31",3.0,16,[-1000,1000],[0,150],[-100,100],"1081",['x1','x2','y1','y2','x3','x4','y3','y4']),
    ("dc32",3.0,16,[-1000,1000],[0,150],[-100,100],"1082",['x1','x2','y1','y2'])]
prm_df = spark.createDataFrame(detprm, columns)

mapping_df = None
tref_mapping_df = None
mapping_dfs = {}
udf_dict = {}

prm_df.show(2,truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/06 15:24:11 WARN Utils: Your hostname, gpuana02, resolves to a loopback address: 127.0.1.1; using 133.11.152.134 instead (on interface enp4s0)
26/02/06 15:24:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/02/06 15:24:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


+----+---------+------+-------------+------------+-----------+-------------+--------------------------------+
|name|cell_size|nwires|valid_time   |charge_range|time_gate  |calib_runname|planes                          |
+----+---------+------+-------------+------------+-----------+-------------+--------------------------------+
|dc31|3.0      |16    |[-1000, 1000]|[0, 150]    |[-100, 100]|1081         |[x1, x2, y1, y2, x3, x4, y3, y4]|
|dc32|3.0      |16    |[-1000, 1000]|[0, 150]    |[-100, 100]|1082         |[x1, x2, y1, y2]                |
+----+---------+------+-------------+------------+-----------+-------------+--------------------------------+



In [3]:

detList = DETECTOR_NAMES
#detector_df = rawDF.select(constants.ID_COLNAME).dropDuplicates([constants.ID_COLNAME])
for mwdc in detList:
      print(prm_df.filter(F.col("name")==mwdc).select("valid_time").collect()[0])
      planes_list = prm_df.filter(F.col("name")==mwdc).select("planes").collect()[0][0]
      NWIRE = prm_df.filter(F.col("name")==mwdc).select("nwires").collect()[0][0]
      for plane in planes_list:
        tname = "dc31_" + plane + "_timing"
        cname = "dc31_" + plane + "_charge"
        iname = "dc31_" + plane + "_id"
        print(prm_df.filter(F.col('name')==mwdc).select('charge_range').collect()[0][0]) 
        print(f"tname:{tname} cname:{cname} iname:{iname}")

Row(valid_time=[-1000, 1000])
[0, 150]
tname:dc31_x1_timing cname:dc31_x1_charge iname:dc31_x1_id
[0, 150]
tname:dc31_x2_timing cname:dc31_x2_charge iname:dc31_x2_id
[0, 150]
tname:dc31_y1_timing cname:dc31_y1_charge iname:dc31_y1_id
[0, 150]
tname:dc31_y2_timing cname:dc31_y2_charge iname:dc31_y2_id
[0, 150]
tname:dc31_x3_timing cname:dc31_x3_charge iname:dc31_x3_id
[0, 150]
tname:dc31_x4_timing cname:dc31_x4_charge iname:dc31_x4_id
[0, 150]
tname:dc31_y3_timing cname:dc31_y3_charge iname:dc31_y3_id
[0, 150]
tname:dc31_y4_timing cname:dc31_y4_charge iname:dc31_y4_id
Row(valid_time=[-1000, 1000])
[0, 150]
tname:dc31_x1_timing cname:dc31_x1_charge iname:dc31_x1_id
[0, 150]
tname:dc31_x2_timing cname:dc31_x2_charge iname:dc31_x2_id
[0, 150]
tname:dc31_y1_timing cname:dc31_y1_charge iname:dc31_y1_id
[0, 150]
tname:dc31_y2_timing cname:dc31_y2_charge iname:dc31_y2_id


In [10]:
#cell_size = prm_df.filter(F.col("name")==mwdc).select("cell_size").collect()[0][0]
cell_size = prm_df.filter(F.col("name")==mwdc).select("cell_size").collect()[0][0]
print(cell_size / 2.)
calib_runname = prm_df.filter(F.col("name")==mwdc).select("calib_runname").collect()[0][0]
print(calib_runname + ".csv")

1.5
1082.csv


In [22]:
columns2=["name", "ns2mm","delayoffset", "linecalib","exchange","reflectx","geometry","TXSumLimit","TYSumLimit"]
detprm2=[["fe7ppac1",[1.240,1.242],[0.92,1.58],[-9.119,2.9301], 0, 0,[ 0.0,0.0,182.8],[-800., 800.],[-800., 800.]],
        ["fe7ppac2",[1.257,1.257],[0.05,0.04],[4.02,0.661],    0, 1,[0.87,0.0,502.8],[-800., 800.],[-800., 800.]]] 
prm_df2 = spark.createDataFrame(detprm2, columns2)
prm_df2.show(2,truncate=False)
temp=prm_df2.filter(F.col("name")=="fe7ppac2")
ns2mm = temp.select("ns2mm").collect()[0][0][0]
if temp.select("reflectx").collect()[0][0]==True:
    print("A")
else:
    print("B")

print(ns2mm)

+--------+--------------+------------+----------------+--------+--------+------------------+---------------+---------------+
|name    |ns2mm         |delayoffset |linecalib       |exchange|reflectx|geometry          |TXSumLimit     |TYSumLimit     |
+--------+--------------+------------+----------------+--------+--------+------------------+---------------+---------------+
|fe7ppac1|[1.24, 1.242] |[0.92, 1.58]|[-9.119, 2.9301]|0       |0       |[0.0, 0.0, 182.8] |[-800.0, 800.0]|[-800.0, 800.0]|
|fe7ppac2|[1.257, 1.257]|[0.05, 0.04]|[4.02, 0.661]   |0       |1       |[0.87, 0.0, 502.8]|[-800.0, 800.0]|[-800.0, 800.0]|
+--------+--------------+------------+----------------+--------+--------+------------------+---------------+---------------+

A
1.257
